In [3]:
def getZippedFilePaths():
    zip_file_names = []
    for dirname, _, filenames in os.walk(r"C:\Users\s234031\Desktop\Flare Project"):
        for filename in filenames:
            if filename.split('.')[-1] == 'zip':
                zip_file_names.append((os.path.join(dirname, filename)))
    return zip_file_names

###########################################################################################
# ----------------------------------------------------------------------------------------
# Preprocess images and mask
# ----------------------------------------------------------------------------------------
##########################################################################################
# file_path = train_hq.zip
def preprocess_image(file_path):
    # Load and decode the image
    img = tf.io.read_file(file_path)
    # You can adjust channels based on your images (3 for RGB)
    img = tf.image.decode_jpeg(img, channels = 3) # Returned as uint8
    # Normalize the pixel values to [0, 1]
    img = tf.image.convert_image_dtype(img, tf.float32)
    # Resize the image to your desired dimensions
    img = tf.image.resize(img, [96, 128], method = 'nearest')
    return img

# file_path = train_masks.zip
def preprocess_target(file_path):
    # Load and decode the image
    mask = tf.io.read_file(file_path)
    # Normalizing to between 0 and 1 (only two classes)
    mask = tf.image.decode_image(mask, expand_animations = False, dtype = tf.float32)
    # Get only one value for the 3rd channel
    mask = tf.math.reduce_max(mask, axis = -1, keepdims = True)
    # Resize the image to your desired dimensions
    mask = tf.image.resize(mask, [96, 128], method = 'nearest')
    return mask

###########################################################################################
# ----------------------------------------------------------------------------------------
# Loss function
# ----------------------------------------------------------------------------------------
##########################################################################################
def dice_coef(y_true, y_pred, smooth=10e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    return dice

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)
###########################################################################################
# ----------------------------------------------------------------------------------------
# Display pred results
# ----------------------------------------------------------------------------------------
##########################################################################################
def display(display_list):
    plt.figure(figsize=(15, 15))
    title = ['Input Image', 'True Mask', 'Predicted Mask']
    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.preprocessing.image.array_to_img(display_list[i]))
        plt.axis('off')
    plt.show()

# Converting probabilities from *.predict(dataset) to the class index
def create_mask(pred_mask):
    mask = pred_mask[..., -1] >= 0.5
    pred_mask[..., -1] = tf.where(mask, 1, 0)
    # Return only first mask of batch
    return pred_mask[0]

# Predict images visualization
def show_predictions(model, dataset = None, num = 1):
    # Displays the first image of each of the num batches
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = model.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)])
    else:
        display([sample_image, sample_mask,
             create_mask(model.predict(sample_image[tf.newaxis, ...]))])

In [ ]:
def analyze_mask_values(mask_files, sample_size = 5, title = "Mask Analysis"):
    # Analyzes the values in masks before processing.
        # mask_files: list of mask file paths
        # sample_size: number of masks to analyze
        # title: output header

    print(f"\n{'='*60}")
    print(f"{title} (analysis {min(sample_size, len(mask_files))} mask)")
    print('='*60)
    
    for i, mask_path in enumerate(mask_files[:sample_size]):
        try:
            # Reading and decoding the source file
            mask_bytes = tf.io.read_file(mask_path)
            mask = tf.image.decode_image(mask_bytes, expand_animations=False, dtype=tf.uint8)
            
            print(f"\Mask {i+1}: {os.path.basename(mask_path)}")
            print(f"  Form: {mask.shape}")
            print(f"  Data type: {mask.dtype}")
            
            # Analyzing unique values
            if len(mask.shape) == 3:
                # If the image is multi-channel
                if mask.shape[2] == 1:
                    values = mask.numpy().flatten()
                else:
                    # For RGB, we look at individual channels
                    values_r = mask.numpy()[..., 0].flatten()
                    values_g = mask.numpy()[..., 1].flatten()
                    values_b = mask.numpy()[..., 2].flatten()
                    
                    print(f"  Unique values in the R channel: {np.unique(values_r)}")
                    print(f"  Unique values in the G channel: {np.unique(values_g)}")
                    print(f"  Unique values in the B channel: {np.unique(values_b)}")
                    
                    # We also look at unique RGB combinations.
                    reshaped = mask.numpy().reshape(-1, mask.shape[2])
                    unique_combinations = np.unique(reshaped, axis=0)
                    print(f"  Unique RGB combinations ({len(unique_combinations)}):")
                    for combo in unique_combinations[:10]:  # Showing the first 10
                        print(f"    {combo}")
                    if len(unique_combinations) > 10:
                        print(f"    ... and more {len(unique_combinations) - 10}")
                    values = np.concatenate([values_r, values_g, values_b])
            else:
                values = mask.numpy().flatten()
            
            unique_values = np.unique(values)
            print(f"  All unique values ({len(unique_values)}): {unique_values}")
            print(f"  Range of values: [{values.min()}, {values.max()}]")
            
            # Histogram of values
            plt.figure(figsize=(10, 3))
            plt.subplot(1, 2, 1)
            plt.hist(values, bins = 50, alpha = 0.7, color = 'blue', edgecolor = 'black')
            plt.title(f'Distribution of values (mask {i+1})')
            plt.xlabel('Pixel Value')
            plt.ylabel('Frequency')
            
            # Mask Visualization
            plt.subplot(1, 2, 2)
            if len(mask.shape) == 3 and mask.shape[2] == 3:
                plt.imshow(mask.numpy())
            else:
                plt.imshow(mask.numpy(), cmap='gray', vmin=0, vmax=255)
            plt.title(f'Mask Visualization {i+1}')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error during analysis {mask_path}: {e}")

def analyze_processed_masks(dataset, sample_size = 5, title = "Analysis of processed masks"):
    # Analyzes the values in masks after processing.
        # dataset: tf.data.Dataset with processed masks
        # sample_size: number of masks to analyze
        # title: output header
    
    print(f"\n{'='*60}")
    print(f"{title}")
    print('='*60)
    
    # We take the first sample_size elements from the dataset
    iterator = iter(dataset)
    for i in range(min(sample_size, 5)):
        try:
            mask = next(iterator)
            
            print(f"\nProcessed mask {i+1}:")
            print(f"  Form: {mask.shape}")
            print(f"  Data type: {mask.dtype}")
            
            # Analysis of unique values
            values = mask.numpy().flatten()
            unique_values = np.unique(values)
            
            print(f"  Unique values ({len(unique_values)}): {unique_values}")
            print(f"  Range of values: [{values.min()}, {values.max()}]")
            
            # Counting the number of each value
            value_counts = {}
            for val in values:
                value_counts[val] = value_counts.get(val, 0) + 1
            
            print("  Distributing:")
            for val in sorted(unique_values):
                count = value_counts.get(val, 0)
                percentage = (count / len(values)) * 100
                print(f"    {val}: {count} pixel values ({percentage:.2f}%)")
            
            # Visualization
            plt.figure(figsize=(10, 3))
            
            # The histogram
            plt.subplot(1, 2, 1)
            bins = len(unique_values) if len(unique_values) <= 50 else 50
            plt.hist(values, bins = bins, alpha=0.7, color='green', edgecolor='black')
            plt.title(f'Distribution of values (processed {i+1})')
            plt.xlabel('Pixel Value')
            plt.ylabel('Frequency')
            
            # Image of the mask
            plt.subplot(1, 2, 2)
            if len(mask.shape) == 3 and mask.shape[2] > 1:
                # If multi-channel
                plt.imshow(mask.numpy())
            else:
                # If it is single-channel
                plt.imshow(mask.numpy(), cmap='viridis')
                plt.colorbar()
            plt.title(f'Visualization of the processed mask {i+1}')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        except StopIteration:
            break
        except Exception as e:
            print(f"Error when analyzing the processed mask {i+1}: {e}")

In [3]:

print(tf.config.list_physical_devices('GPU'))

[]


In [2]:
import tensorflow as tf

def encoder_block(inputs, num_filters):

    x = tf.keras.layers.Conv2D(num_filters, 3, padding='valid')(inputs)
    x = tf.keras.layers.Activation('relu')(x)
    
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='valid')(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(x)
    
    return x
def decoder_block(inputs, skip_features, num_filters):

    x = tf.keras.layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='valid')(inputs)

    skip_features = tf.keras.layers.Resizing(x.shape[1], x.shape[2])(skip_features)

    x = tf.keras.layers.Concatenate()([x, skip_features])

    x = tf.keras.layers.Conv2D(num_filters, 3, padding='valid')(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(num_filters, 3, padding='valid')(x)
    x = tf.keras.layers.Activation('relu')(x)

    return x
def unet_model(input_shape=(256, 256, 3), num_classes=3):
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # Contracting Path (Encoder)
    s1 = encoder_block(inputs, 64)
    s2 = encoder_block(s1, 128)
    s3 = encoder_block(s2, 256)
    s4 = encoder_block(s3, 512)
    
    # Bottleneck
    b1 = tf.keras.layers.Conv2D(1024, 3, padding='valid')(s4)
    b1 = tf.keras.layers.Activation('relu')(b1)
    b1 = tf.keras.layers.Conv2D(1024, 3, padding='valid')(b1)
    b1 = tf.keras.layers.Activation('relu')(b1)
    
    # Expansive Path (Decoder)
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    
    outputs = tf.keras.layers.Conv2D(num_classes, 1, padding='valid', activation='sigmoid')(d4)
    
    model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name='U-Net')
    return model

if __name__ == '__main__':
    model = unet_model(input_shape=(572, 572, 3), num_classes=3)
    model.summary()

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 572, 572,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 570, 570,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 570, 570,  │          0 │ conv2d[0][0]      │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 568, 568,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 568, 568,  │          0 │ conv2d_1[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 284, 284,  │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 282, 282,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 282, 282,  │          0 │ conv2d_2[0][0]    │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 280, 280,  │    147,584 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 280, 280,  │          0 │ conv2d_3[0][0]    │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 140, 140,  │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 138, 138,  │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 138, 138,  │          0 │ conv2d_4[0][0]    │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 136, 136,  │    590,080 │ activation_4[0][… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 136, 136,  │          0 │ conv2d_5[0][0]    │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 68, 68,    │          0 │ activation_5[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 66, 66,    │  1,180,160 │ max_pooling2d_2[

 Total params: 31,031,875 (118.38 MB)

 Trainable params: 31,031,875 (118.38 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import os
import zipfile
zip_file_names = getZippedFilePaths()

items_to_remove = ['/kaggle/input/carvana-image-masking-challenge/train.zip', 
                   '/kaggle/input/carvana-image-masking-challenge/test.zip',
                   'C:\\Users\\s234031\\Desktop\\Flare Project\\f_s_1_1500.v2-add-1.tfrecord (1).zip']
     
zip_file_names = [item for item in zip_file_names if item not in items_to_remove]
extract_dir = zip_file_names.with_suffix("")  # same name, no .zip
for zip_file_path in zip_file_names:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
print(f"✅ Extracted to: {extract_dir}")

AttributeError: 'list' object has no attribute 'with_suffix'

In [6]:
import zipfile
from pathlib import Path

zip_file_names = getZippedFilePaths()

items_to_remove = [
    '/kaggle/input/carvana-image-masking-challenge/train.zip',
    '/kaggle/input/carvana-image-masking-challenge/test.zip',
    r'C:\Users\s234031\Desktop\Flare Project\f_s_1_1500.v2-add-1.tfrecord (1).zip'
]

# Filter ZIP files
zip_file_names = [z for z in zip_file_names if z not in items_to_remove]

for zip_file_path in zip_file_names:
    zip_path = Path(zip_file_path)

    # Create extraction directory (same name, no .zip)
    extract_dir = zip_path.with_suffix("")

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"✅ Extracted: {zip_path.name}")
    print(f"   → {extract_dir}")

✅ Extracted: f_s_1_1500.v2-add-1.coco-segmentation.zip
   → C:\Users\s234031\Desktop\Flare Project\f_s_1_1500.v2-add-1.coco-segmentation


In [8]:
from pathlib import Path

root = Path(r"C:\Users\s234031\Desktop\Flare Project\f_s_1_1500.v2-add-1.coco-segmentation")

for path in root.rglob("_annotations.coco.json"):
    print("✅ Found:", path)

✅ Found: C:\Users\s234031\Desktop\Flare Project\f_s_1_1500.v2-add-1.coco-segmentation\train\_annotations.coco.json


In [9]:
from pycocotools.coco import COCO

ANN_PATH = extract_dir /"train"/ "_annotations.coco.json"
coco = COCO(str(ANN_PATH))

print("Images:", len(coco.getImgIds()))
print("Categories:", coco.getCatIds())

loading annotations into memory...
Done (t=0.11s)
creating index...
index created!
Images: 1483
Categories: [0, 1, 2]


In [10]:
import numpy as np

def coco_to_semantic_mask(image_id):
    img_info = coco.loadImgs(image_id)[0]
    height, width = img_info["height"], img_info["width"]

    mask = np.zeros((height, width), dtype=np.uint8)

    ann_ids = coco.getAnnIds(imgIds=image_id, iscrowd=False)
    anns = coco.loadAnns(ann_ids)

    for ann in anns:
        class_id = ann["category_id"]   # 0=bg,1=fire,2=smoke
        instance_mask = coco.annToMask(ann)
        mask[instance_mask == 1] = class_id

    return mask

In [14]:
def analyze_mask_values(mask_files, sample_size = 5, title = "Mask Analysis"):
    # Analyzes the values in masks before processing.
        # mask_files: list of mask file paths
        # sample_size: number of masks to analyze
        # title: output header

    print(f"\n{'='*60}")
    print(f"{title} (analysis {min(sample_size, len(mask_files))} mask)")
    print('='*60)
    
    for i, mask_path in enumerate(mask_files[:sample_size]):
        try:
            # Reading and decoding the source file
            mask_bytes = tf.io.read_file(mask_path)
            mask = tf.image.decode_image(mask_bytes, expand_animations=False, dtype=tf.uint8)
            
            print(f"\Mask {i+1}: {os.path.basename(mask_path)}")
            print(f"  Form: {mask.shape}")
            print(f"  Data type: {mask.dtype}")
            
            # Analyzing unique values
            if len(mask.shape) == 3:
                # If the image is multi-channel
                if mask.shape[2] == 1:
                    values = mask.numpy().flatten()
                else:
                    # For RGB, we look at individual channels
                    values_r = mask.numpy()[..., 0].flatten()
                    values_g = mask.numpy()[..., 1].flatten()
                    values_b = mask.numpy()[..., 2].flatten()
                    
                    print(f"  Unique values in the R channel: {np.unique(values_r)}")
                    print(f"  Unique values in the G channel: {np.unique(values_g)}")
                    print(f"  Unique values in the B channel: {np.unique(values_b)}")
                    
                    # We also look at unique RGB combinations.
                    reshaped = mask.numpy().reshape(-1, mask.shape[2])
                    unique_combinations = np.unique(reshaped, axis=0)
                    print(f"  Unique RGB combinations ({len(unique_combinations)}):")
                    for combo in unique_combinations[:10]:  # Showing the first 10
                        print(f"    {combo}")
                    if len(unique_combinations) > 10:
                        print(f"    ... and more {len(unique_combinations) - 10}")
                    values = np.concatenate([values_r, values_g, values_b])
            else:
                values = mask.numpy().flatten()
            
            unique_values = np.unique(values)
            print(f"  All unique values ({len(unique_values)}): {unique_values}")
            print(f"  Range of values: [{values.min()}, {values.max()}]")
            
            # Histogram of values
            plt.figure(figsize=(10, 3))
            plt.subplot(1, 2, 1)
            plt.hist(values, bins = 50, alpha = 0.7, color = 'blue', edgecolor = 'black')
            plt.title(f'Distribution of values (mask {i+1})')
            plt.xlabel('Pixel Value')
            plt.ylabel('Frequency')
            
            # Mask Visualization
            plt.subplot(1, 2, 2)
            if len(mask.shape) == 3 and mask.shape[2] == 3:
                plt.imshow(mask.numpy())
            else:
                plt.imshow(mask.numpy(), cmap='gray', vmin=0, vmax=255)
            plt.title(f'Mask Visualization {i+1}')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error during analysis {mask_path}: {e}")

def analyze_processed_masks(dataset, sample_size = 5, title = "Analysis of processed masks"):
    # Analyzes the values in masks after processing.
        # dataset: tf.data.Dataset with processed masks
        # sample_size: number of masks to analyze
        # title: output header
    
    print(f"\n{'='*60}")
    print(f"{title}")
    print('='*60)
    
    # We take the first sample_size elements from the dataset
    iterator = iter(dataset)
    for i in range(min(sample_size, 5)):
        try:
            mask = next(iterator)
            
            print(f"\nProcessed mask {i+1}:")
            print(f"  Form: {mask.shape}")
            print(f"  Data type: {mask.dtype}")
            
            # Analysis of unique values
            values = mask.numpy().flatten()
            unique_values = np.unique(values)
            
            print(f"  Unique values ({len(unique_values)}): {unique_values}")
            print(f"  Range of values: [{values.min()}, {values.max()}]")
            
            # Counting the number of each value
            value_counts = {}
            for val in values:
                value_counts[val] = value_counts.get(val, 0) + 1
            
            print("  Distributing:")
            for val in sorted(unique_values):
                count = value_counts.get(val, 0)
                percentage = (count / len(values)) * 100
                print(f"    {val}: {count} pixel values ({percentage:.2f}%)")
            
            # Visualization
            plt.figure(figsize=(10, 3))
            
            # The histogram
            plt.subplot(1, 2, 1)
            bins = len(unique_values) if len(unique_values) <= 50 else 50
            plt.hist(values, bins = bins, alpha=0.7, color='green', edgecolor='black')
            plt.title(f'Distribution of values (processed {i+1})')
            plt.xlabel('Pixel Value')
            plt.ylabel('Frequency')
            
            # Image of the mask
            plt.subplot(1, 2, 2)
            if len(mask.shape) == 3 and mask.shape[2] > 1:
                # If multi-channel
                plt.imshow(mask.numpy())
            else:
                # If it is single-channel
                plt.imshow(mask.numpy(), cmap='viridis')
                plt.colorbar()
            plt.title(f'Visualization of the processed mask {i+1}')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        except StopIteration:
            break
        except Exception as e:
            print(f"Error when analyzing the processed mask {i+1}: {e}")

In [1]:
from pycocotools.coco import COCO
print("✅ pycocotools is working")
coco = COCO("_annotations.coco.json")

✅ pycocotools is working
loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: '_annotations.coco.json'

In [3]:

import sys
print(sys.executable)
print(sys.version)


c:\Users\s234031\AppData\Local\miniconda3\envs\tf\python.exe
3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]
